# Duka Electronics: H1 2026 Sales Exploration
**Analyst:** Stanley Wanyoike Ngugi

 Half-year review, January to June 2026

## Executive Summary
- Duka Electronics earned KES 71.8M from 1,000 orders between 1 January and 29 June 2026, an average of KES 71,793 per order.
- Nairobi is the strongest branch with KES 26.1M (36.4% of revenue) from 376 orders, and branches rank identically by revenue and by orders because average order values only range from KES 68,353 to KES 75,591.
- Laptops generate KES 41.5M (57.9% of revenue) from 639 units, while tablets sell the most units (691) but earn only KES 19.3M.
- Weekends account for 27.7% of orders (277) and 27.8% of revenue, and average order revenue is almost identical (KES 71,972 on weekends versus KES 71,724 on weekdays), so there is no weekend premium.
- April was the best month (KES 13.5M) and June the worst (KES 10.9M), a 24.0% gap, while the 83 Premium online orders in Nairobi and Mombasa brought in KES 10.2M (14.2% of revenue).

In [1]:
import os
if not os.path.exists("generate_data.py"):
    !git clone https://github.com/SWNgugi/plp-da-a3-duka-sales-and-agent-cleanup.git
    %cd plp-da-a3-duka-sales-and-agent-cleanup

import numpy as np
import pandas as pd

if not os.path.exists("duka_sales.csv"):
    !python generate_data.py
os.makedirs("outputs", exist_ok=True)
os.makedirs("excel", exist_ok=True)

fatal: destination path 'plp-da-a3-duka-sales-and-agent-cleanup' already exists and is not an empty directory.
/content/plp-da-a3-duka-sales-and-agent-cleanup


## Task 1: First-five-minutes inspection

In [2]:
df = pd.read_csv("duka_sales.csv", parse_dates=["order_date"])
display(df.head())
display(df.tail())
print("Shape:", df.shape)
df.info()
display(df.describe())
print(df.nunique())
print(df["branch"].value_counts(normalize=True).round(3))
print(df["channel"].value_counts(normalize=True).round(3))
print("Period:", df["order_date"].min(), "to", df["order_date"].max())

,order_id,order_date,branch,product,quantity,channel,unit_price
0,1001,2026-06-03,Nakuru,Phone,3,In-store,15000
1,1002,2026-02-02,Nairobi,Laptop,4,In-store,65000
2,1003,2026-01-05,Nakuru,Tablet,5,In-store,28000
3,1004,2026-04-26,Nairobi,Headphones,5,In-store,2500
4,1005,2026-03-07,Nairobi,Headphones,3,In-store,2500


,order_id,order_date,branch,product,quantity,channel,unit_price
995,1996,2026-03-04,Mombasa,Laptop,5,In-store,65000
996,1997,2026-03-06,Nairobi,Tablet,2,Phone order,28000
997,1998,2026-01-20,Nairobi,Laptop,3,Online,65000
998,1999,2026-03-02,Nairobi,Headphones,2,In-store,2500
999,2000,2026-01-30,Nakuru,Charger,4,In-store,1200


Shape: (1000, 7)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   order_id    1000 non-null   int64         
 1   order_date  1000 non-null   datetime64[ns]
 2   branch      1000 non-null   object        
 3   product     1000 non-null   object        
 4   quantity    1000 non-null   int64         
 5   channel     1000 non-null   object        
 6   unit_price  1000 non-null   int64         
dtypes: datetime64[ns](1), int64(3), object(3)
memory usage: 54.8+ KB


,order_id,order_date,quantity,unit_price
count,1000.000000,1000,1000.000000,1000.000000
mean,1500.500000,2026-03-30 08:44:09.600000,3.012000,23918.300000
min,1001.000000,2026-01-01 00:00:00,1.000000,1200.000000
25%,1250.750000,2026-02-14 00:00:00,2.000000,2500.000000
50%,1500.500000,2026-03-29 00:00:00,3.000000,15000.000000
75%,1750.250000,2026-05-14 00:00:00,4.000000,28000.000000
max,2000.000000,2026-06-29 00:00:00,5.000000,65000.000000
std,288.819436,NaN,1.416991,23908.790546


order_id      1000
order_date     179
branch           5
product          5
quantity         5
channel          3
unit_price       5
dtype: int64
branch
Nairobi    0.376
Mombasa    0.186
Nakuru     0.162
Kisumu     0.159
Eldoret    0.117
Name: proportion, dtype: float64
channel
In-store       0.579
Online         0.320
Phone order    0.101
Name: proportion, dtype: float64
Period: 2026-01-01 00:00:00 to 2026-06-29 00:00:00


**Observations**
- **Shape:** 1,000 rows × 7 columns; `order_id` is unique, so each row is one order.
- **Types:** `order_date` is parsed as datetime thanks to `parse_dates`; `quantity` and `unit_price` are integers; `branch`, `product` and `channel` are text categories.
- **Missing values:** none in any column.
- **Date range:** 1 January to 29 June 2026, which matches the half-year review period.
- **Categories:** 5 branches, 5 products and 3 channels. Nairobi has 37.6% of orders; channels split 57.9% in-store, 32.0% online and 10.1% phone orders.
- **Describe:** quantity ranges from 1 to 5 and unit price from KES 1,200 to 65,000, so product price differences, not quantities, will drive revenue.

## Task 2: Derived columns
`revenue = quantity × unit_price`. Month and weekday come from the `.dt` accessor. `price_band` is Premium for a unit price of KES 28,000 or more, otherwise Value.

In [3]:
df["revenue"] = df["quantity"] * df["unit_price"]
df["month"] = df["order_date"].dt.month_name()
df["weekday"] = df["order_date"].dt.day_name()
df["is_weekend"] = df["order_date"].dt.dayofweek >= 5
df["price_band"] = np.where(df["unit_price"] >= 28000, "Premium", "Value")

print(df["price_band"].value_counts())
print(df["is_weekend"].value_counts())
print("Total revenue: KES", f"{df['revenue'].sum():,}")
df[["revenue", "month", "weekday", "is_weekend", "price_band"]].head()

price_band
Value      565
Premium    435
Name: count, dtype: int64
is_weekend
False    723
True     277
Name: count, dtype: int64
Total revenue: KES 71,792,500


,revenue,month,weekday,is_weekend,price_band
0,45000,June,Wednesday,False,Value
1,260000,February,Monday,False,Premium
2,140000,January,Monday,False,Premium
3,12500,April,Sunday,True,Value
4,7500,March,Saturday,True,Value


## Task 3: Branch ranking by revenue and by orders

In [4]:
by_branch = df.groupby("branch").agg(total_revenue=("revenue", "sum"), n_orders=("revenue", "size"))
by_branch["revenue_rank"] = by_branch["total_revenue"].rank(ascending=False).astype(int)
by_branch["orders_rank"] = by_branch["n_orders"].rank(ascending=False).astype(int)
by_branch["avg_order_value"] = (by_branch["total_revenue"] / by_branch["n_orders"]).round(0)
by_branch["revenue_share_%"] = (by_branch["total_revenue"] / by_branch["total_revenue"].sum() * 100).round(1)
by_branch.sort_values("total_revenue", ascending=False)

,total_revenue,n_orders,revenue_rank,orders_rank,avg_order_value,revenue_share_%
branch,,,,,,
Nairobi,26106100,376,1,1,69431.0,36.4
Mombasa,14003700,186,2,2,75289.0,19.5
Nakuru,11970400,162,3,3,73891.0,16.7
Kisumu,10868100,159,4,4,68353.0,15.1
Eldoret,8844200,117,5,5,75591.0,12.3


**Answer:** Yes, the rankings are the same: Nairobi, Mombasa, Nakuru, Kisumu, Eldoret by both revenue and orders. Orders are assigned to branches independently of product and quantity, so every branch gets a similar product mix and its average order value stays in a narrow band (KES 68,353 to 75,591). With similar order values, order volume decides revenue. Nairobi leads with 376 orders and KES 26.1M (36.4%). Eldoret has the highest average order value (KES 75,591) but ranks last because it has the fewest orders (117). Nakuru and Kisumu are close (162 vs 159 orders), so their order could easily flip with different data.

## Task 4: Top three products by revenue and by units

In [5]:
by_product = df.groupby("product").agg(revenue=("revenue", "sum"), units=("quantity", "sum"))
print("Top 3 by revenue:\n", by_product.nlargest(3, "revenue"))
print("\nTop 3 by units:\n", by_product.nlargest(3, "units"))

Top 3 by revenue:
           revenue  units
product                 
Laptop   41535000    639
Tablet   19348000    691
Phone     8880000    592

Top 3 by units:
           revenue  units
product                 
Tablet   19348000    691
Laptop   41535000    639
Phone     8880000    592


**Answer:** The same three products appear in both lists (Laptop, Tablet, Phone) but in a different order. By revenue: Laptop KES 41.5M, Tablet KES 19.3M, Phone KES 8.9M. By units: Tablet 691, Laptop 639, Phone 592. Tablets sell more units than laptops, but a laptop costs KES 65,000 against KES 28,000, so laptops earn more than twice the revenue and make up 57.9% of the total. These three products bring in 97.2% of revenue, while headphones and chargers add little (KES 1.4M and KES 0.6M) despite similar unit counts.

## Task 5: Weekend vs weekday orders

In [6]:
df.groupby("is_weekend")["revenue"].agg(n_orders="size", total_revenue="sum", avg_order_revenue="mean").round(0)

,n_orders,total_revenue,avg_order_revenue
is_weekend,,,
False,723,51856200,71724.0
True,277,19936300,71972.0


**Answer:** Weekdays: 723 orders, KES 51,856,200, average KES 71,724. Weekends: 277 orders, KES 19,936,300, average KES 71,972. Weekends make up 27.7% of orders and 27.8% of revenue, and the average order is only 0.3% higher, so weekend customers spend almost exactly the same as weekday customers. There is no weekend premium to exploit.

## Task 6: Best and worst month

In [7]:
order = ["January", "February", "March", "April", "May", "June"]
monthly = (df.groupby("month")
             .agg(revenue=("revenue", "sum"), orders=("revenue", "size"), avg_order=("revenue", "mean"))
             .reindex(order).round(0))
print(monthly)

best, worst = monthly["revenue"].idxmax(), monthly["revenue"].idxmin()
pct_diff = (monthly.loc[best, "revenue"] - monthly.loc[worst, "revenue"]) / monthly.loc[worst, "revenue"] * 100
print(f"Best: {best} KES {monthly.loc[best, 'revenue']:,.0f} | Worst: {worst} KES {monthly.loc[worst, 'revenue']:,.0f} | Difference: {pct_diff:.1f}% (relative to {worst})")

print(df[df["product"] == "Laptop"].groupby("month").size().reindex(order))

           revenue  orders  avg_order
month                                
January   11786600     173    68131.0
February  11772000     159    74038.0
March     12396800     186    66649.0
April     13518300     167    80948.0
May       11415400     153    74610.0
June      10903400     162    67305.0
Best: April KES 13,518,300 | Worst: June KES 10,903,400 | Difference: 24.0% (relative to June)
month
January     34
February    39
March       35
April       43
May         34
June        35
dtype: int64


**Answer:** April is the best month (KES 13,518,300) and June the worst (KES 10,903,400). April is 24.0% higher than June, or equivalently June is 19.3% lower than April. The gap is not about order counts (167 in April versus 162 in June) but about order value: April's average order was KES 80,948 against KES 67,305 in June, helped by 43 laptop orders versus 35. Revenue is driven by how many high-priced items are sold, so a month with the most orders (March, 186) is not necessarily the best month.

## Task 7: Premium online orders from Nairobi and Mombasa

In [8]:
mask = (df["price_band"] == "Premium") & (df["channel"] == "Online") & (df["branch"].isin(["Nairobi", "Mombasa"]))
premium = df[mask]
print(len(premium), "orders, KES", f"{premium['revenue'].sum():,}")
print(f"Share of total revenue: {premium['revenue'].sum() / df['revenue'].sum():.1%}")
print(premium["branch"].value_counts())
premium.to_csv("outputs/premium_online_coast_capital.csv", index=False)

83 orders, KES 10,218,000
Share of total revenue: 14.2%
branch
Nairobi    50
Mombasa    33
Name: count, dtype: int64


**Answer:** There are 83 Premium online orders from Nairobi (50) and Mombasa (33), worth KES 10,218,000 in total, or 14.2% of all revenue. Premium here means tablets and laptops, since both have a unit price of KES 28,000 or more. The orders are exported to `outputs/premium_online_coast_capital.csv`.

## Task 8: Spreadsheet cross-check
The Excel SUMIFS table and the PivotTable are compared with the pandas totals.

In [9]:
xl = pd.read_excel("excel/duka_branch_summary.xlsx", sheet_name="Branch_Summary")
xl_totals = xl.set_index("branch")["total_revenue"]
pandas_totals = df.groupby("branch")["revenue"].sum()
print("Branch totals match pandas:", np.allclose(pandas_totals.sort_index(), xl_totals.sort_index()))

pv = pd.read_excel("excel/duka_branch_summary.xlsx", sheet_name="Pivot", header=None)
grand = pv[pv[0].astype(str).str.contains("Grand Total")].iloc[0].dropna().iloc[-1]
print("PivotTable grand total:", f"{grand:,.0f}", "| pandas total:", f"{df['revenue'].sum():,}")
print("Pivot matches pandas:", np.isclose(grand, df["revenue"].sum()))

Branch totals match pandas: True
PivotTable grand total: 71,792,500 | pandas total: 71,792,500
Pivot matches pandas: True
